In [7]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.vectorstores import FAISS
from langchain_ollama.chat_models import ChatOllama
from langchain_core.documents import Document
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_ollama.embeddings import OllamaEmbeddings

In [ ]:
# Recreate the document objects from the previous data
docs = [
    Document(
        page_content=(
            """The Grand Canyon is one of the most visited natural wonders in the world.
Photosynthesis is the process by which green plants convert sunlight into energy.
Millions of tourists travel to see it every year. The rocks date back millions of years."""
        ),
        metadata={"source": "Doc1"},
    ),
    Document(
        page_content=(
            """In medieval Europe, castles were built primarily for defense.
The chlorophyll in plant cells captures sunlight during photosynthesis.
Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
        ),
        metadata={"source": "Doc2"},
    ),
    Document(
        page_content=(
            """Basketball was invented by Dr. James Naismith in the late 19th century.
It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
        ),
        metadata={"source": "Doc3"},
    ),
    Document(
        page_content=(
            """<first line partially cut off>
Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells."""
        ),
        metadata = {"source"  : 'Doc4'}
    ),
]

In [8]:
embeddings = OllamaEmbeddings(model ='nomic-embed-text')
vectorstore = FAISS.from_documents(docs,embeddings)

In [12]:
retriver = vectorstore.as_retriever(kwargs= {'k': 5})

In [11]:
llm = ChatOllama(model = 'llama3.2:3b')
compressor = LLMChainExtractor.from_llm(llm)

In [14]:
## Create the contextial compression retriver
compression_retriver =ContextualCompressionRetriever(
    base_retriever= retriver,
    base_compressor= compressor
)

In [15]:
query = 'what is photosynthesis'
compressed_results = compression_retriver.invoke(query)

In [17]:
for i, docs in enumerate(compressed_results):
    print(f'{i+1} : {docs}')

1 : page_content='> Photosynthesis does not occur in animal cells.' metadata={'source': 'Doc4'}
2 : page_content='The extracted relevant parts of the context are:

> The chlorophyll in plant cells captures sunlight during photosynthesis.' metadata={'source': 'Doc2'}
